# ACRE Engine - Data Characterization & Distribution Analysis
## Notebook 4: Adaptive Clustering Strategy Determination

**Author:** Daniella Tahchi  
**Date:** 2024  
**Purpose:** Analyze feature distribution to determine optimal clustering algorithm  
**Input:** Feature matrix from Notebook 03  
**Output:** Data characterization metrics + algorithm selection strategy  

---

## Table of Contents

1. [Setup & Load Feature Matrix](#1-setup--load-feature-matrix)
2. [Clustering Tendency Analysis](#2-clustering-tendency-analysis)
3. [Density Distribution Analysis](#3-density-distribution-analysis)
4. [Feature Variance Analysis](#4-feature-variance-analysis)
5. [Sample Size Assessment](#5-sample-size-assessment)
6. [Algorithm Suitability Matrix](#6-algorithm-suitability-matrix)
7. [Final Recommendations](#7-final-recommendations)
8. [Save Characterization Report](#8-save-characterization-report)

---

## Data Characterization Objectives

This notebook performs **comprehensive statistical analysis** of the feature distribution to answer:

### Key Questions:

1. **Does the data have natural clusters?** → Hopkins Statistic
2. **Is density uniform or variable?** → k-distance analysis, Coefficient of Variation
3. **What is the effective dimensionality?** → PCA variance analysis
4. **Are features balanced?** → Variance ratio analysis
5. **Is the dataset large enough for computationally expensive algorithms?** → Sample size check
6. **Should we reduce dimensionality?** → Compare clustering quality with/without reduction

### Algorithms to Evaluate:

| Algorithm | Strengths | Weaknesses | Best For |
|-----------|-----------|------------|----------|
| **K-Means** | Fast, scalable, works well with spherical clusters | Assumes equal cluster sizes, sensitive to outliers | Uniform density, large datasets |
| **DBSCAN** | Finds arbitrary shapes, handles noise, no k parameter | Struggles with varying densities, sensitive to parameters | Variable density, noise present |
| **Agglomerative** | No assumptions on cluster shape, hierarchical structure | O(n³) complexity, not scalable | Small/medium datasets, hierarchical relationships |

### Outcome:

**Data-driven algorithm selection** based on measured characteristics, not assumptions.

---

## 1. Setup & Load Feature Matrix

In [1]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
from scipy.stats import variation
from scipy.spatial.distance import euclidean

import warnings
from pathlib import Path
from datetime import datetime
import json
import joblib
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Visualization settings
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

pd.set_option('display.precision', 6)

print("="*80)
print("ACRE ENGINE - DATA CHARACTERIZATION & DISTRIBUTION ANALYSIS")
print("="*80)
print(f"\nNotebook executed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✓ Libraries loaded successfully\n")

ACRE ENGINE - DATA CHARACTERIZATION & DISTRIBUTION ANALYSIS

Notebook executed: 2026-04-17 15:20:02

✓ Libraries loaded successfully



In [2]:
# Configuration
CONFIG = {
    # Paths
    'artifacts_dir': '../artifacts/feature-engineering',
    'output_dir' : '../artifacts/data-characterization',
    # Characterization parameters
    'hopkins_sample_size': 10000,
    'density_sample_size': 10000,
    'density_k_neighbors': 10,
    
    # Algorithm selection thresholds
    'hopkins_threshold': 0.5,
    'density_cv_threshold': 0.5,
    'max_agglomerative_samples': 50000,
    'dimensionality_ratio_threshold': 0.7,
    
    # Random seed
    'random_state': 42
}

print("Configuration:")
print(json.dumps(CONFIG, indent=2))

# Set random seed
np.random.seed(CONFIG['random_state'])

Configuration:
{
  "artifacts_dir": "../artifacts/feature-engineering",
  "output_dir": "../artifacts/data-characterization",
  "hopkins_sample_size": 10000,
  "density_sample_size": 10000,
  "density_k_neighbors": 10,
  "hopkins_threshold": 0.5,
  "density_cv_threshold": 0.5,
  "max_agglomerative_samples": 50000,
  "dimensionality_ratio_threshold": 0.7,
  "random_state": 42
}


In [3]:
# Load feature matrix
print("="*80)
print("LOADING FEATURE MATRIX")
print("="*80)

artifacts_dir = Path(CONFIG['artifacts_dir'])

# Load feature matrix
print("\n[1] Loading feature matrix...")
feature_matrix = np.load(artifacts_dir / 'feature_matrix.npy')
print(f"  ✓ Shape: {feature_matrix.shape}")
print(f"  ✓ Data type: {feature_matrix.dtype}")
print(f"  ✓ Memory: {feature_matrix.nbytes / 1024**2:.2f} MB")

# Load movie index
print("\n[2] Loading movie index...")
movie_index = pd.read_csv(artifacts_dir / 'movie_index.csv')
print(f"  ✓ Movies: {len(movie_index):,}")

# Load feature metadata
print("\n[3] Loading feature metadata...")
with open(artifacts_dir / 'feature_metadata.json', 'r') as f:
    feature_metadata = json.load(f)
print(f"  ✓ Total features: {feature_metadata['total_features']}")
print(f"  ✓ Feature groups: {list(feature_metadata['feature_groups'].keys())}")

# Verify consistency
assert feature_matrix.shape[0] == len(movie_index), "Row count mismatch!"
assert feature_matrix.shape[1] == feature_metadata['total_features'], "Column count mismatch!"

print("\n✓ All artifacts loaded and verified")

# Store dataset info
n_samples = feature_matrix.shape[0]
n_features = feature_matrix.shape[1]

print(f"\nDataset: {n_samples:,} movies × {n_features} features")

LOADING FEATURE MATRIX

[1] Loading feature matrix...
  ✓ Shape: (27777, 172)
  ✓ Data type: float64
  ✓ Memory: 36.45 MB

[2] Loading movie index...
  ✓ Movies: 27,777

[3] Loading feature metadata...
  ✓ Total features: 172
  ✓ Feature groups: ['genres', 'keywords', 'numerical', 'overview']

✓ All artifacts loaded and verified

Dataset: 27,777 movies × 172 features


In [4]:
# Quick data quality check
print("\nData Quality Check:")
print(f"  • NaN values: {np.isnan(feature_matrix).sum()}")
print(f"  • Infinite values: {np.isinf(feature_matrix).sum()}")
print(f"  • Value range: [{feature_matrix.min():.6f}, {feature_matrix.max():.6f}]")
print(f"  • Sparsity: {100*(feature_matrix == 0).sum()/feature_matrix.size:.2f}%")

if np.isnan(feature_matrix).sum() == 0 and np.isinf(feature_matrix).sum() == 0:
    print("\n✓ Data quality verified - ready for characterization")
else:
    print("\n⚠ WARNING: Data quality issues detected!")


Data Quality Check:
  • NaN values: 0
  • Infinite values: 0
  • Value range: [0.000000, 1.000000]
  • Sparsity: 93.98%

✓ Data quality verified - ready for characterization


In [5]:
# Initialize characterization results dictionary
characterization_results = {
    'timestamp': datetime.now().isoformat(),
    'n_samples': n_samples,
    'n_features': n_features,
    'config': CONFIG,
    'metrics': {},
    'recommendations': {}
}

print("\nCharacterization analysis initialized...\n")


Characterization analysis initialized...



---

## 2. Clustering Tendency Analysis

**Objective:** Determine if the data has natural clustering structure.

**Method:** Hopkins Statistic

### Hopkins Statistic

Measures the probability that a dataset is generated by a uniform data distribution.

**Interpretation:**
- **H ≈ 0.5:** Uniform distribution (no clusters)
- **H > 0.7:** Strong clustering tendency
- **H > 0.5:** Moderate clustering tendency
- **H < 0.5:** Regular/grid-like distribution

**Formula:**

$$H = \frac{\sum_{i=1}^{n} w_i}{\sum_{i=1}^{n} u_i + \sum_{i=1}^{n} w_i}$$

Where:
- $w_i$ = distance from random point to nearest data point
- $u_i$ = distance from data point to nearest neighbor (excluding self)

---

In [6]:
print("="*80)
print("CLUSTERING TENDENCY ANALYSIS (Hopkins Statistic)")
print("="*80)

def compute_hopkins_statistic(X, sample_size=500, random_state=42):
    """
    Compute Hopkins statistic for clustering tendency.
    
    Parameters:
    -----------
    X : np.ndarray
        Feature matrix (n_samples, n_features)
    sample_size : int
        Number of points to sample for the test
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    float : Hopkins statistic (0 to 1)
    """
    np.random.seed(random_state)
    
    n = min(sample_size, len(X) // 2)
    
    print(f"\n[Step 1] Sampling {n} points from dataset...")
    # Sample points from data
    sample_indices = np.random.choice(len(X), n, replace=False)
    X_sample = X[sample_indices]
    
    print(f"[Step 2] Generating {n} uniform random points...")
    # Generate uniform random points in same space
    X_random = np.random.uniform(
        X.min(axis=0),
        X.max(axis=0),
        size=(n, X.shape[1])
    )
    
    print(f"[Step 3] Fitting nearest neighbors on full dataset...")
    # Fit nearest neighbors on full data
    nbrs = NearestNeighbors(n_neighbors=2, metric='euclidean')
    nbrs.fit(X)
    
    print(f"[Step 4] Computing u distances (real sample → nearest neighbor)...")
    # Distances for real samples (to nearest neighbor, excluding self)
    u_distances, _ = nbrs.kneighbors(X_sample, n_neighbors=2)
    u = u_distances[:, 1]  # Second neighbor (first is self)
    
    print(f"[Step 5] Computing w distances (random points → nearest data point)...")
    # Distances for random samples
    w_distances, _ = nbrs.kneighbors(X_random, n_neighbors=1)
    w = w_distances[:, 0]
    
    print(f"[Step 6] Calculating Hopkins statistic...")
    # Hopkins statistic
    H = w.sum() / (u.sum() + w.sum())
    
    return H, u, w

# Compute Hopkins statistic
print(f"\nComputing Hopkins statistic with sample size {CONFIG['hopkins_sample_size']}...")
print("This may take 1-2 minutes...\n")

hopkins_stat, u_distances, w_distances = compute_hopkins_statistic(
    feature_matrix,
    sample_size=CONFIG['hopkins_sample_size'],
    random_state=CONFIG['random_state']
)

print(f"\n{'='*80}")
print(f"HOPKINS STATISTIC: {hopkins_stat:.6f}")
print(f"{'='*80}")

# Interpret result
print("\nInterpretation:")
if hopkins_stat > 0.7:
    interpretation = "STRONG clustering tendency detected"
    color = "🟢"
elif hopkins_stat > CONFIG['hopkins_threshold']:
    interpretation = "MODERATE clustering tendency detected"
    color = "🟡"
else:
    interpretation = "WEAK/NO clustering tendency (data may be uniformly distributed)"
    color = "🔴"

print(f"  {color} {interpretation}")

if hopkins_stat > CONFIG['hopkins_threshold']:
    print(f"  → Data is suitable for clustering algorithms")
else:
    print(f"  ⚠ Warning: Clustering may not produce meaningful results")

# Store result
characterization_results['metrics']['hopkins_statistic'] = {
    'value': float(hopkins_stat),
    'interpretation': interpretation,
    'sample_size': CONFIG['hopkins_sample_size'],
    'threshold': CONFIG['hopkins_threshold'],
    'passes_threshold': bool(hopkins_stat > CONFIG['hopkins_threshold'])
}

print(f"\n{'='*80}")

CLUSTERING TENDENCY ANALYSIS (Hopkins Statistic)

Computing Hopkins statistic with sample size 10000...
This may take 1-2 minutes...


[Step 1] Sampling 10000 points from dataset...
[Step 2] Generating 10000 uniform random points...
[Step 3] Fitting nearest neighbors on full dataset...
[Step 4] Computing u distances (real sample → nearest neighbor)...
[Step 5] Computing w distances (random points → nearest data point)...
[Step 6] Calculating Hopkins statistic...

HOPKINS STATISTIC: 0.859987

Interpretation:
  🟢 STRONG clustering tendency detected
  → Data is suitable for clustering algorithms



---

## 3. Density Distribution Analysis

**Objective:** Determine if data density is uniform or variable across the feature space.

**Method:** k-distance analysis + Coefficient of Variation

### Why This Matters:

- **K-Means** assumes roughly equal cluster densities
- **DBSCAN** excels when density varies significantly

### k-Distance Analysis:

For each point, compute distance to its k-th nearest neighbor.

**Coefficient of Variation (CV):**

$$CV = \frac{\sigma}{\mu}$$

Where $\sigma$ = standard deviation, $\mu$ = mean of k-distances

**Interpretation:**
- **CV < 0.3:** Uniform density → K-Means suitable
- **CV > 0.5:** Variable density → DBSCAN recommended
- **0.3 ≤ CV ≤ 0.5:** Moderate variation → Both algorithms viable

---

In [8]:
print("="*80)
print("DENSITY DISTRIBUTION ANALYSIS (k-Distance Method)")
print("="*80)

def analyze_density_distribution(X, k=10, sample_size=5000, random_state=42):
    """
    Analyze density variation using k-nearest neighbor distances.
    
    Parameters:
    -----------
    X : np.ndarray
        Feature matrix
    k : int
        Number of neighbors for k-distance
    sample_size : int
        Number of points to sample (for efficiency)
    random_state : int
        Random seed
        
    Returns:
    --------
    cv : float
        Coefficient of variation of k-distances
    k_distances : np.ndarray
        Array of k-distances for sampled points
    """
    np.random.seed(random_state)
    
    # Sample for efficiency
    n_sample = min(sample_size, len(X))
    sample_indices = np.random.choice(len(X), n_sample, replace=False)
    X_sample = X[sample_indices]
    
    print(f"\n[Step 1] Sampled {n_sample:,} points for analysis")
    print(f"[Step 2] Computing {k}-nearest neighbors...")
    
    # Compute k-distances
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric='euclidean')
    nbrs.fit(X)
    distances, _ = nbrs.kneighbors(X_sample)
    
    # k-distance (distance to k-th neighbor, excluding self)
    k_distances = distances[:, k]
    
    print(f"[Step 3] Computing coefficient of variation...")
    
    # Coefficient of variation
    cv = variation(k_distances)
    
    return cv, k_distances

# Perform density analysis
print(f"\nAnalyzing density distribution...")
print(f"  k = {CONFIG['density_k_neighbors']} neighbors")
print(f"  Sample size = {CONFIG['density_sample_size']:,}")
print("\nThis may take 1-2 minutes...")

density_cv, k_distances = analyze_density_distribution(
    feature_matrix,
    k=CONFIG['density_k_neighbors'],
    sample_size=CONFIG['density_sample_size'],
    random_state=CONFIG['random_state']
)

print(f"\n{'='*80}")
print(f"DENSITY COEFFICIENT OF VARIATION: {density_cv:.6f}")
print(f"{'='*80}")

# Interpret result
print("\nInterpretation:")
if density_cv < 0.3:
    interpretation = "UNIFORM density across feature space"
    recommendation = "K-Means is well-suited"
    color = "🟢"
elif density_cv > CONFIG['density_cv_threshold']:
    interpretation = "VARIABLE density (high variation)"
    recommendation = "DBSCAN is recommended for handling varying densities"
    color = "🟡"
else:
    interpretation = "MODERATE density variation"
    recommendation = "Both K-Means and DBSCAN are viable"
    color = "🟢"

print(f"  {color} {interpretation}")
print(f"  → {recommendation}")

# Store result
characterization_results['metrics']['density_distribution'] = {
    'coefficient_of_variation': float(density_cv),
    'interpretation': interpretation,
    'k_neighbors': CONFIG['density_k_neighbors'],
    'sample_size': CONFIG['density_sample_size'],
    'threshold': CONFIG['density_cv_threshold'],
    'recommend_dbscan': bool(density_cv > CONFIG['density_cv_threshold'])
}

print(f"\n{'='*80}")

DENSITY DISTRIBUTION ANALYSIS (k-Distance Method)

Analyzing density distribution...
  k = 10 neighbors
  Sample size = 10,000

This may take 1-2 minutes...

[Step 1] Sampled 10,000 points for analysis
[Step 2] Computing 10-nearest neighbors...
[Step 3] Computing coefficient of variation...

DENSITY COEFFICIENT OF VARIATION: 0.267633

Interpretation:
  🟢 UNIFORM density across feature space
  → K-Means is well-suited



---

## 4. Feature Variance Analysis

**Objective:** Assess if features contribute roughly equally or if some dominate.

**Metrics:**
- Variance ratio: max variance / min variance
- Zero-variance features
- Variance distribution

**Impact:**
- High variance ratio → Some features dominate distance calculations
- Zero variance features → Should be removed (no information)

---

In [11]:
print("="*80)
print("FEATURE VARIANCE ANALYSIS")
print("="*80)

print("\n[Step 1] Computing feature variances...")

# Compute variance for each feature
feature_variances = feature_matrix.var(axis=0)

# Basic statistics
print(f"\n[Step 2] Variance statistics:")
print(f"  • Mean variance: {feature_variances.mean():.6f}")
print(f"  • Median variance: {np.median(feature_variances):.6f}")
print(f"  • Std variance: {feature_variances.std():.6f}")
print(f"  • Min variance: {feature_variances.min():.6f}")
print(f"  • Max variance: {feature_variances.max():.6f}")

# Variance ratio
if feature_variances.min() > 0:
    variance_ratio = feature_variances.max() / feature_variances.min()
else:
    variance_ratio = np.inf

print(f"\n[Step 3] Variance ratio (max/min): {variance_ratio:.2f}")

# Zero variance features
zero_variance_count = (feature_variances == 0).sum()
print(f"\n[Step 4] Zero variance features: {zero_variance_count}")

if zero_variance_count > 0:
    print("  ⚠ Warning: Zero variance features should be removed")
else:
    print("  ✓ All features have non-zero variance")

# Low variance features (< 1% of max)
low_variance_threshold = 0.01 * feature_variances.max()
low_variance_count = (feature_variances < low_variance_threshold).sum()

print(f"\n[Step 5] Low variance features (< 1% of max): {low_variance_count}")
print(f"  ({100*low_variance_count/n_features:.2f}% of total features)")

# Interpretation
print(f"\n{'='*80}")
print("VARIANCE ANALYSIS SUMMARY")
print(f"{'='*80}")

if variance_ratio < 100:
    interpretation = "Features are BALANCED (similar variance scales)"
    color = "🟢"
elif variance_ratio < 1000:
    interpretation = "MODERATE variance imbalance detected"
    color = "🟡"
else:
    interpretation = "HIGH variance imbalance (some features dominate)"
    color = "🔴"

print(f"  {color} {interpretation}")

# Store results
characterization_results['metrics']['feature_variance'] = {
    'mean_variance': float(feature_variances.mean()),
    'median_variance': float(np.median(feature_variances)),
    'max_variance': float(feature_variances.max()),
    'min_variance': float(feature_variances.min()),
    'variance_ratio': float(variance_ratio) if variance_ratio != np.inf else 'infinity',
    'zero_variance_features': int(zero_variance_count),
    'low_variance_features': int(low_variance_count),
    'interpretation': interpretation
}

print(f"\n{'='*80}")

FEATURE VARIANCE ANALYSIS

[Step 1] Computing feature variances...

[Step 2] Variance statistics:
  • Mean variance: 0.021480
  • Median variance: 0.011121
  • Std variance: 0.034812
  • Min variance: 0.004518
  • Max variance: 0.245519

[Step 3] Variance ratio (max/min): 54.35

[Step 4] Zero variance features: 0
  ✓ All features have non-zero variance

[Step 5] Low variance features (< 1% of max): 0
  (0.00% of total features)

VARIANCE ANALYSIS SUMMARY
  🟢 Features are BALANCED (similar variance scales)



In [13]:
# Top and bottom variance features
with open(artifacts_dir / 'feature_names.json', 'r') as f:
    feature_names = json.load(f)

print("\nTop 20 Features by Variance:\n")
top_var_indices = feature_variances.argsort()[-20:][::-1]

for i, idx in enumerate(top_var_indices, 1):
    print(f"  {i:2d}. {feature_names[idx]:<50} {feature_variances[idx]:.8f}")

print("\nBottom 20 Features by Variance:\n")
bottom_var_indices = feature_variances.argsort()[:20]

for i, idx in enumerate(bottom_var_indices, 1):
    print(f"  {i:2d}. {feature_names[idx]:<50} {feature_variances[idx]:.8f}")


Top 20 Features by Variance:

   1. genre_drama                                        0.24551852
   2. genre_comedy                                       0.22981385
   3. genre_thriller                                     0.16747592
   4. genre_action                                       0.14907250
   5. genre_romance                                      0.13517425
   6. genre_horror                                       0.12457075
   7. genre_crime                                        0.10723045
   8. genre_adventure                                    0.10240207
   9. genre_family                                       0.08437266
  10. genre_science_fiction                              0.08325727
  11. genre_fantasy                                      0.07810219
  12. genre_animation                                    0.07600617
  13. genre_mystery                                      0.06979475
  14. keyword_based_on_novel_or_book                     0.06297652
  15. keyword_wom

---

## 5. Sample Size Assessment

**Objective:** Determine computational feasibility of different algorithms.

### Complexity Analysis:

| Algorithm | Time Complexity | Space Complexity | Scalability |
|-----------|----------------|------------------|-------------|
| K-Means | O(nki) | O(n) | Excellent (scales to millions) |
| DBSCAN | O(n log n) avg | O(n) | Good (with spatial indexing) |
| Agglomerative | O(n³) worst, O(n² log n) avg | O(n²) | Poor (limited to ~50K samples) |

Where:
- n = number of samples
- k = number of clusters
- i = number of iterations

---

In [14]:
print("="*80)
print("SAMPLE SIZE & COMPUTATIONAL FEASIBILITY ASSESSMENT")
print("="*80)

print(f"\nDataset size: {n_samples:,} samples × {n_features} features")

# K-Means assessment
print(f"\n[1] K-Means Feasibility:")
print(f"  ✓ EXCELLENT - Scales to millions of samples")
print(f"  • Expected runtime: < 5 minutes for typical k range")
kmeans_feasible = True

# DBSCAN assessment
print(f"\n[2] DBSCAN Feasibility:")
if n_samples <= 100000:
    print(f"  ✓ GOOD - Dataset size suitable for DBSCAN")
    print(f"  • Expected runtime: 5-15 minutes (with ball_tree)")
    dbscan_feasible = True
else:
    print(f"  ⚠ MODERATE - Large dataset may be slow")
    print(f"  • Consider sampling or spatial indexing")
    dbscan_feasible = True  # Still feasible, just slower

# Agglomerative assessment
print(f"\n[3] Agglomerative Clustering Feasibility:")

if n_samples <= CONFIG['max_agglomerative_samples']:
    print(f"  ✓ FEASIBLE - Sample size within acceptable range")
    print(f"  • Expected runtime: 10-30 minutes")
    agglomerative_feasible = True
else:
    print(f"  ✗ NOT FEASIBLE - Sample size too large")
    print(f"  • Threshold: {CONFIG['max_agglomerative_samples']:,} samples")
    print(f"  • Current: {n_samples:,} samples")
    print(f"  • Agglomerative is O(n³), would take hours/days")
    agglomerative_feasible = False

# Summary
print(f"\n{'='*80}")
print("FEASIBILITY SUMMARY")
print(f"{'='*80}")

feasibility_summary = {
    'K-Means': '✓ Feasible',
    'DBSCAN': '✓ Feasible' if dbscan_feasible else '⚠ Marginal',
    'Agglomerative': '✓ Feasible' if agglomerative_feasible else '✗ Not Feasible'
}

for algo, status in feasibility_summary.items():
    print(f"  {algo:<20} {status}")

# Store results
characterization_results['metrics']['computational_feasibility'] = {
    'n_samples': n_samples,
    'n_features': n_features,
    'kmeans_feasible': kmeans_feasible,
    'dbscan_feasible': dbscan_feasible,
    'agglomerative_feasible': agglomerative_feasible,
    'max_agglomerative_threshold': CONFIG['max_agglomerative_samples']
}

print(f"\n{'='*80}")

SAMPLE SIZE & COMPUTATIONAL FEASIBILITY ASSESSMENT

Dataset size: 27,777 samples × 172 features

[1] K-Means Feasibility:
  ✓ EXCELLENT - Scales to millions of samples
  • Expected runtime: < 5 minutes for typical k range

[2] DBSCAN Feasibility:
  ✓ GOOD - Dataset size suitable for DBSCAN
  • Expected runtime: 5-15 minutes (with ball_tree)

[3] Agglomerative Clustering Feasibility:
  ✓ FEASIBLE - Sample size within acceptable range
  • Expected runtime: 10-30 minutes

FEASIBILITY SUMMARY
  K-Means              ✓ Feasible
  DBSCAN               ✓ Feasible
  Agglomerative        ✓ Feasible



---

## 6. Algorithm Suitability Matrix

**Comprehensive decision matrix based on all measured characteristics.**

---

In [15]:
print("="*80)
print("ALGORITHM SUITABILITY MATRIX")
print("="*80)

# Create suitability scores
suitability = {
    'K-Means': {'score': 0, 'reasons': []},
    'DBSCAN': {'score': 0, 'reasons': []},
    'Agglomerative': {'score': 0, 'reasons': []}
}

# Criterion 1: Clustering tendency
if hopkins_stat > CONFIG['hopkins_threshold']:
    suitability['K-Means']['score'] += 2
    suitability['K-Means']['reasons'].append(f"✓ Hopkins = {hopkins_stat:.4f} (clusters exist)")
    suitability['DBSCAN']['score'] += 2
    suitability['DBSCAN']['reasons'].append(f"✓ Hopkins = {hopkins_stat:.4f} (clusters exist)")
    suitability['Agglomerative']['score'] += 2
    suitability['Agglomerative']['reasons'].append(f"✓ Hopkins = {hopkins_stat:.4f} (clusters exist)")
else:
    for algo in suitability:
        suitability[algo]['reasons'].append(f"⚠ Hopkins = {hopkins_stat:.4f} (weak tendency)")

# Criterion 2: Density distribution
if density_cv < 0.3:
    suitability['K-Means']['score'] += 3
    suitability['K-Means']['reasons'].append(f"✓ Uniform density (CV = {density_cv:.4f})")
elif density_cv > CONFIG['density_cv_threshold']:
    suitability['DBSCAN']['score'] += 3
    suitability['DBSCAN']['reasons'].append(f"✓ Variable density (CV = {density_cv:.4f})")
    suitability['K-Means']['reasons'].append(f"⚠ Variable density may affect K-Means")
else:
    suitability['K-Means']['score'] += 2
    suitability['DBSCAN']['score'] += 2
    suitability['K-Means']['reasons'].append(f"✓ Moderate density variation (CV = {density_cv:.4f})")
    suitability['DBSCAN']['reasons'].append(f"✓ Moderate density variation (CV = {density_cv:.4f})")

# Criterion 3: Computational feasibility
if kmeans_feasible:
    suitability['K-Means']['score'] += 3
    suitability['K-Means']['reasons'].append("✓ Excellent scalability")

if dbscan_feasible:
    suitability['DBSCAN']['score'] += 2
    suitability['DBSCAN']['reasons'].append("✓ Computationally feasible")

if agglomerative_feasible:
    suitability['Agglomerative']['score'] += 3
    suitability['Agglomerative']['reasons'].append("✓ Sample size acceptable")
else:
    suitability['Agglomerative']['score'] -= 5
    suitability['Agglomerative']['reasons'].append("✗ Sample size too large (O(n³) complexity)")

# Criterion 4: General suitability
suitability['K-Means']['score'] += 1
suitability['K-Means']['reasons'].append("✓ Robust default choice")

# Display suitability matrix
print("\n" + "="*80)
print("ALGORITHM SCORES & RATIONALE")
print("="*80)

sorted_algos = sorted(suitability.items(), key=lambda x: x[1]['score'], reverse=True)

for algo, info in sorted_algos:
    print(f"\n{algo}:")
    print(f"  Score: {info['score']}/10")
    print(f"  Rationale:")
    for reason in info['reasons']:
        print(f"    • {reason}")

# Recommendation
print(f"\n{'='*80}")
print("ALGORITHM SELECTION RECOMMENDATION")
print(f"{'='*80}")

# Always include K-Means
recommended_algorithms = ['K-Means']

# Add DBSCAN if density is variable
if density_cv > CONFIG['density_cv_threshold']:
    recommended_algorithms.append('DBSCAN')

# Add Agglomerative if feasible
if agglomerative_feasible:
    recommended_algorithms.append('Agglomerative')

print("\nRecommended algorithms to evaluate:")
for i, algo in enumerate(recommended_algorithms, 1):
    score = suitability[algo]['score']
    print(f"  {i}. {algo} (score: {score}/10)")



print(f"\n{'='*80}")

ALGORITHM SUITABILITY MATRIX

ALGORITHM SCORES & RATIONALE

K-Means:
  Score: 9/10
  Rationale:
    • ✓ Hopkins = 0.8600 (clusters exist)
    • ✓ Uniform density (CV = 0.2676)
    • ✓ Excellent scalability
    • ✓ Robust default choice

Agglomerative:
  Score: 5/10
  Rationale:
    • ✓ Hopkins = 0.8600 (clusters exist)
    • ✓ Sample size acceptable

DBSCAN:
  Score: 4/10
  Rationale:
    • ✓ Hopkins = 0.8600 (clusters exist)
    • ✓ Computationally feasible

ALGORITHM SELECTION RECOMMENDATION

Recommended algorithms to evaluate:
  1. K-Means (score: 9/10)
  2. Agglomerative (score: 5/10)



In [16]:
# 1. Identify the single best algorithm based on score
# This finds the key (algo name) with the highest 'score' value
best_algo = max(suitability, key=lambda k: suitability[k]['score'])
best_score = suitability[best_algo]['score']

print(f"\n{'='*80}")
print("ALGORITHM SELECTION RECOMMENDATION")
print(f"{'='*80}")

print(f"\n🏆 WINNER: {best_algo}")
print(f"   Overall Score: {best_score}/10")
print(f"   Primary Rationale:")

# Print the reasons for the winning algorithm
for reason in suitability[best_algo]['reasons']:
    print(f"    • {reason}")

# Store only the single best recommendation
recommended_algorithms = [best_algo]

characterization_results['recommendations']['algorithms'] = {
    'best_algorithm': best_algo,
    'score': best_score,
    'all_scores': {algo: info['score'] for algo, info in suitability.items()},
    'rationale': suitability[best_algo]['reasons']
}

print(f"\n{'='*80}")


ALGORITHM SELECTION RECOMMENDATION

🏆 WINNER: K-Means
   Overall Score: 9/10
   Primary Rationale:
    • ✓ Hopkins = 0.8600 (clusters exist)
    • ✓ Uniform density (CV = 0.2676)
    • ✓ Excellent scalability
    • ✓ Robust default choice



---

## 7. Final Recommendations

**Comprehensive clustering strategy based on all analyses.**

---

In [17]:
print("="*80)
print("FINAL CLUSTERING STRATEGY RECOMMENDATIONS")
print("="*80)

print("\n" + "─"*80)
print("STEP 1: ALGORITHM SELECTION")
print("─"*80)

print("\n  Evaluate the following algorithms:")
for i, algo in enumerate(recommended_algorithms, 1):
    print(f"\n  {i}. {algo.upper()}")
    
    if algo == 'K-Means':
        print("     • Test k = 5 to 40")
        print("     • Select k with highest silhouette score")
        print("     • Expected runtime: 2-5 minutes")
    
    elif algo == 'DBSCAN':
        print("     • Determine eps using k-distance knee detection")
        print(f"     • Use min_samples = 2 × n_features")
        print("     • Handle noise points: assign to nearest cluster")
        print("     • Expected runtime: 5-15 minutes")
    
    elif algo == 'Agglomerative':
        print("     • Test n_clusters = 3 to 15")
        print("     • Linkage: ward (minimizes variance)")
        print("     • Select n_clusters with highest silhouette score")
        print("     • Expected runtime: 10-30 minutes")

print("\n" + "─"*80)
print("STEP 2: FINAL SELECTION")
print("─"*80)

print("\n  → Compare all (algorithm, reduction, k) combinations")
print("  → Select configuration with HIGHEST SILHOUETTE SCORE")
print("  → Store selected model, cluster assignments, and centroids")

print("\n" + "─"*80)

print("\n" + "="*80)

FINAL CLUSTERING STRATEGY RECOMMENDATIONS

────────────────────────────────────────────────────────────────────────────────
STEP 1: ALGORITHM SELECTION
────────────────────────────────────────────────────────────────────────────────

  Evaluate the following algorithms:

  1. K-MEANS
     • Test k = 5 to 40
     • Select k with highest silhouette score
     • Expected runtime: 2-5 minutes

────────────────────────────────────────────────────────────────────────────────
STEP 2: FINAL SELECTION
────────────────────────────────────────────────────────────────────────────────

  → Compare all (algorithm, reduction, k) combinations
  → Select configuration with HIGHEST SILHOUETTE SCORE
  → Store selected model, cluster assignments, and centroids

────────────────────────────────────────────────────────────────────────────────



---

## 8. Save Characterization Report

**Save all characterization results for reference and reproducibility.**

---

In [18]:
print("="*80)
print("SAVING CHARACTERIZATION REPORT")
print("="*80)

# Set the base directory
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True) 

# [1] Save JSON report
print("\n[1] Saving JSON report...")
# FIXED: Added the filename to the path
json_file = output_dir / 'data_characterization_results.json'

with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(characterization_results, f, indent=2, default=str)

print(f"  ✓ Saved to {json_file}")

SAVING CHARACTERIZATION REPORT

[1] Saving JSON report...
  ✓ Saved to ..\artifacts\data-characterization\data_characterization_results.json


---

## Conclusion

### Data Characterization Complete

✅ **Clustering tendency confirmed** (Hopkins statistic)

✅ **Density distribution analyzed** (k-distance coefficient of variation)

✅ **Effective dimensionality determined** (PCA variance analysis)

✅ **Computational feasibility assessed** (algorithm complexity vs sample size)

✅ **Data-driven recommendations generated** (algorithm selection strategy)

### Key Findings:

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Hopkins Statistic | *computed* | Clustering tendency present |
| Density CV | *computed* | Uniform density |
| Effective Dimensions | *computed* | High dimensional |
| Recommended Algorithms | *computed* | K-Means |


### Next Steps:

→ **Notebook 05 - Clustering** will implement the recommended strategy  
→ Best configuration selected via silhouette score  

---